# HBICP iteration 2 on Colab
Runtime > Change runtime type > GPU (T4). Run the cells in order. If Colab disconnects, reopen and run all again: it resumes from `last.pt` on Drive.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
REPO = 'https://github.com/Alessio-Pecilli/Hamiltonian-Blind-In-Context-Propagator.git'
if not os.path.exists('/content/hbicp'):
    subprocess.run(['git', 'clone', REPO, '/content/hbicp'], check=True)
else:
    subprocess.run(['git', '-C', '/content/hbicp', 'pull'], check=True)
%cd /content/hbicp/Hamiltonian-Blind-In-Context-Propagator
!pip install -q h5py pyyaml scipy
# datasets, checkpoints and results live on Drive so they survive disconnects
D = '/content/drive/MyDrive/hbicp'
for d in ['datasets', 'runs', 'results']:
    os.makedirs(f'{D}/{d}', exist_ok=True)
    if not os.path.islink(d): os.symlink(f'{D}/{d}', d)
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/hbicp/Hamiltonian-Blind-In-Context-Propagator
True Tesla T4


In [5]:
# datasets (skipped if already on Drive)
!test -f datasets/n8/test.h5 || bash scripts/make_datasets.sh local 2

In [6]:
# same logic as scripts/run_iter2.ps1: run A (reference) then B (pauli_A), idempotent
import os, subprocess, torch
steps = 8000
runs = [('n8_ref', []), ('n8_pauliA', ['model.pauli_A=true'])]
def step(d):
    p = f'{d}/last.pt'
    return torch.load(p, map_location='cpu', weights_only=False)['step'] if os.path.exists(p) else 0
for name, extra in runs:
    d, res = f'runs/{name}', f'results/{name}/results.json'
    if os.path.exists(res): continue
    os.makedirs(d, exist_ok=True)
    if step(d) < steps:
        a = ['python', 'train.py', '--preset', 'local', '--set', f'train.steps={steps}', 'train.seed=0', f'train.out_dir={d}'] + extra
        if os.path.exists(f'{d}/last.pt'): a += ['--resume', f'{d}/last.pt']
        subprocess.run(a, check=True)
    if step(d) >= steps:
        subprocess.run(['python', '-m', 'eval.evaluate', '--ckpt', f'{d}/last.pt', '--data', 'datasets/n8/test.h5',
                        '--T', '8', '16', '32', '64', '--horizon', '64',
                        '--baselines', 'persistence', 'dmd', 'dmd4', 'oracle', 'hl_oracle', '--out', f'results/{name}'], check=True)

## Publish
One-time setup: GitHub > Settings > Developer settings > Fine-grained tokens > only this repo, Contents: Read and write. In Colab, left sidebar key icon > Add secret `GH_TOKEN` with the token, enable Notebook access.

In [7]:
# publish to the dashboard (git push) from Colab; needs Colab secret GH_TOKEN (see markdown above)
import os, shutil, subprocess
from google.colab import userdata
tok = userdata.get('GH_TOKEN')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://{tok}@github.com/Alessio-Pecilli/Hamiltonian-Blind-In-Context-Propagator.git'], check=True)
subprocess.run(['git', 'config', 'user.name', 'Alessio Pecilli'], check=True)
subprocess.run(['git', 'config', 'user.email', 'alessiopecilli@gmail.com'], check=True)
jobs = [('n8_ref', ['n8_ref'], 'n8 A riferimento (no probe, no opzioni coppie, S off)',
         'Run A: riferimento. pair_attn/triangle/pair_latent spenti, nessun L_probe, use_S=false, n=8, 8000 step, seed 0, validazione ogni 1000 step.'),
        ('n8_pauliA', ['n8_ref', 'n8_pauliA'], 'n8 B pauli_A (un generatore di Pauli per tutto A)',
         'Run B: come A ma model.pauli_A=true. Stesso seed e stessi step di A. L export contiene anche A per il confronto.')]
for name, keep, label, notes in jobs:
    marker = f'results/{name}/.published'
    if os.path.exists(marker) or not os.path.exists(f'results/{name}/results.json'): continue
    shutil.rmtree('stage_publish', ignore_errors=True)
    for k in keep:
        os.makedirs(f'stage_publish/runs/{k}'); os.makedirs(f'stage_publish/results/{k}')
        for f in ['config.json', 'log.jsonl', 'last.pt']: shutil.copy(f'runs/{k}/{f}', f'stage_publish/runs/{k}')
        shutil.copy(f'results/{k}/results.json', f'stage_publish/results/{k}')
    subprocess.run(['git', 'pull', '--rebase', '-q'], check=True)
    r = subprocess.run(['python', '-m', 'tools.publish_dashboard', '--label', label, '--notes', notes, '--runs', 'stage_publish/runs',
        '--results', 'stage_publish/results', '--data', 'datasets/n8/test.h5', '--ckpt', f'runs/{name}/last.pt',
        '--T', '32', '--horizon', '64', '--sample', '0', '--keep', '100'])
    if r.returncode == 0: open(marker, 'w').close()
    shutil.rmtree('stage_publish', ignore_errors=True)
